<a href="https://colab.research.google.com/github/Frans1828/scikit-learn-cookbook/blob/main/13.%20Deploying%20Models%20in%20Production/ch13_deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## Identitas Mahasiswa

| Informasi | Keterangan |
|---|---|
| **Nama** | Fransisco Sitepu |
| **NIM** | 101032300004 |
| **Kelas** | TK 47 03 |
| **Mata Kuliah** | Machine Learning |
| **Institusi** | Telkom University |

---


# Chapter 13: Deploying Models in Production

## Dari Eksperimen ke Produksi

Setelah model ML berhasil dilatih dan dievaluasi, langkah selanjutnya adalah **deployment** — membuat model tersedia untuk digunakan oleh aplikasi atau pengguna nyata.

### Tantangan Deployment

1. **Reproducibility**: memastikan model yang sama dapat direproduksi
2. **Versioning**: melacak versi model, data, dan kode
3. **Serving**: menyediakan prediksi dengan latensi rendah dan throughput tinggi
4. **Monitoring**: memantau performa model setelah deployment
5. **Pipeline consistency**: memastikan preprocessing yang sama digunakan saat training dan inference

## Model Serialization

Model ML perlu disimpan ke disk setelah training dan dimuat kembali saat inference:

- **joblib**: library Python yang efisien untuk serialisasi objek NumPy/scikit-learn
- **pickle**: built-in Python serialization (lebih lambat untuk array besar)

Pentingnya serialisasi **seluruh pipeline** (bukan hanya model):

```
Training: raw data → Pipeline(preprocessor + model) → predictions
Inference: raw data → Pipeline.predict() → predictions
```

Jika hanya model yang disimpan (tanpa preprocessor), maka preprocessing harus diulang secara manual, yang rawan kesalahan.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import os
import time
from pathlib import Path

from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

# Load dataset
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
feature_names = cancer.feature_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f'Dataset: Breast Cancer Wisconsin')
print(f'Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}')
print(f'Features: {X_train.shape[1]}')


## 13.1 Building a Production-Ready Pipeline

Pipeline yang baik untuk produksi mencakup:
1. **Data validation**: cek tipe dan range data
2. **Preprocessing**: imputation, scaling, encoding
3. **Model**: estimator dengan hyperparameter optimal
4. **Post-processing**: threshold tuning untuk klasifikasi


In [ ]:
# Build and train production pipeline
production_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8,
        random_state=42
    ))
])

# Train
production_pipeline.fit(X_train, y_train)

# Evaluate
train_acc = production_pipeline.score(X_train, y_train)
test_acc = production_pipeline.score(X_test, y_test)
auc = roc_auc_score(y_test, production_pipeline.predict_proba(X_test)[:, 1])

print('=== Production Pipeline Performance ===')
print(f'Training Accuracy: {train_acc:.4f}')
print(f'Test Accuracy:     {test_acc:.4f}')
print(f'Test AUC-ROC:      {auc:.4f}')
print(f'\nDetailed Report:')
print(classification_report(y_test, production_pipeline.predict(X_test),
                           target_names=cancer.target_names))


## 13.2 Model Serialization dengan joblib

`joblib` sangat efisien untuk menyimpan model scikit-learn yang berisi array NumPy besar (seperti Random Forest dengan ribuan pohon).

```python
# Save model
joblib.dump(model, 'model.joblib')

# Load model
loaded_model = joblib.load('model.joblib')
```

**Best Practices**:
- Simpan **seluruh pipeline** (bukan hanya model)
- Sertakan versi scikit-learn dan Python dalam metadata
- Gunakan compression untuk file besar
- Buat unit test untuk memverifikasi prediksi model yang dimuat


In [ ]:
# Save pipeline to disk
save_dir = Path('.') / 'saved_models'
save_dir.mkdir(exist_ok=True)

model_path = save_dir / 'breast_cancer_pipeline_v1.joblib'

# Save with compression level 3 (good balance of speed vs size)
joblib.dump(production_pipeline, model_path, compress=3)

file_size = os.path.getsize(model_path)
print(f'Model saved to: {model_path}')
print(f'File size: {file_size/1024:.1f} KB')

# Load model back
loaded_pipeline = joblib.load(model_path)

# Verify predictions are identical
y_pred_original = production_pipeline.predict(X_test)
y_pred_loaded = loaded_pipeline.predict(X_test)
identical = np.array_equal(y_pred_original, y_pred_loaded)
print(f'\nPredictions identical after load: {identical}')
print(f'Loaded model accuracy: {accuracy_score(y_test, y_pred_loaded):.4f}')

# Benchmark inference time
n_samples = 1000
X_benchmark = np.tile(X_test, (n_samples // len(X_test) + 1, 1))[:n_samples]

start = time.time()
batch_pred = loaded_pipeline.predict(X_benchmark)
batch_time = time.time() - start

print(f'\nInference benchmark:')
print(f'  Batch size: {n_samples}')
print(f'  Total time: {batch_time*1000:.2f} ms')
print(f'  Per sample: {batch_time/n_samples*1000:.4f} ms')


## 13.3 Model Versioning dan Metadata

Dalam produksi, penting untuk melacak metadata model:
- Waktu training
- Versi scikit-learn dan Python
- Performa pada validation set
- Nama fitur dan preprocessing parameters


In [ ]:
import json
import sklearn
import sys
from datetime import datetime

# Create model bundle with metadata
model_metadata = {
    'model_name': 'BreastCancerClassifier',
    'version': '1.0.0',
    'trained_at': datetime.now().isoformat(),
    'python_version': sys.version,
    'sklearn_version': sklearn.__version__,
    'training_samples': X_train.shape[0],
    'test_samples': X_test.shape[0],
    'n_features': X_train.shape[1],
    'feature_names': list(feature_names),
    'target_names': list(cancer.target_names),
    'performance': {
        'test_accuracy': float(test_acc),
        'test_auc_roc': float(auc),
        'train_accuracy': float(train_acc)
    },
    'hyperparameters': production_pipeline['classifier'].get_params()
}

# Save metadata as JSON
meta_path = save_dir / 'breast_cancer_pipeline_v1_metadata.json'
with open(meta_path, 'w') as f:
    json.dump(model_metadata, f, indent=2, default=str)

print('=== Model Metadata ===')
print(f'Model: {model_metadata["model_name"]} v{model_metadata["version"]}')
print(f'Trained: {model_metadata["trained_at"]}')
print(f'scikit-learn: {model_metadata["sklearn_version"]}')
print(f'Performance: accuracy={model_metadata["performance"]["test_accuracy"]:.4f}, AUC={model_metadata["performance"]["test_auc_roc"]:.4f}')
print(f'Metadata saved to: {meta_path}')


## 13.4 Simulating Production Inference

Dalam produksi, data baru masuk satu per satu atau dalam batch. Pipeline yang telah disimpan memastikan preprocessing yang konsisten.


In [ ]:
# Simulate production inference
def predict_single_sample(pipeline, raw_features, feature_names, class_names):
    """Simulate single-sample inference as would happen in an API."""
    X_new = np.array(raw_features).reshape(1, -1)
    prediction = pipeline.predict(X_new)[0]
    probability = pipeline.predict_proba(X_new)[0]
    return {
        'predicted_class': class_names[prediction],
        'predicted_class_idx': int(prediction),
        'probability_malignant': float(probability[0]),
        'probability_benign': float(probability[1]),
        'confidence': float(max(probability))
    }

# Test with a few samples from test set
print('=== Production Inference Simulation ===')
print(f'{"Sample":<8} {"True Label":<15} {"Predicted":<15} {"Confidence":>12} {"Correct?"}')
print('-' * 65)

for i in range(5):
    result = predict_single_sample(
        loaded_pipeline,
        X_test[i],
        feature_names,
        cancer.target_names
    )
    true_label = cancer.target_names[y_test[i]]
    correct = '✓' if result['predicted_class'] == true_label else '✗'
    print(f"{i+1:<8} {true_label:<15} {result['predicted_class']:<15} {result['confidence']:>12.4f} {correct}")

# Batch inference
print(f'\n=== Batch Inference ===')
batch_results = loaded_pipeline.predict_proba(X_test)
print(f'Batch shape: {batch_results.shape}')
print(f'Predictions distribution: {pd.Series(loaded_pipeline.predict(X_test)).value_counts().to_dict()}')

# Clean up saved files
import shutil
shutil.rmtree(save_dir)
print(f'\n(Saved model files cleaned up)')


In [ ]:
# Model comparison: final production model selection
models_final = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(C=10, max_iter=5000, random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('model', RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42))
    ]),
    'Gradient Boosting': Pipeline([
        ('scaler', StandardScaler()),
        ('model', GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42))
    ])
}

print('=== Final Model Selection ===')
print(f'{"Model":<25} {"Test Acc":>10} {"AUC":>8} {"5-CV Acc":>10} {"Inference (ms/sample)":>22}')
print('-' * 78)

for name, model in models_final.items():
    model.fit(X_train, y_train)
    test_acc = model.score(X_test, y_test)
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    cv_acc = cross_val_score(model, X, y, cv=5).mean()

    start = time.time()
    for _ in range(100):
        _ = model.predict(X_test[:1])
    infer_time = (time.time() - start) / 100 * 1000

    print(f'{name:<25} {test_acc:>10.4f} {auc:>8.4f} {cv_acc:>10.4f} {infer_time:>22.3f}')


## Ringkasan: Key Takeaways Chapter 13

| Konsep | Poin Kunci |
|---|---|
| **Pipeline Serialization** | Selalu simpan **seluruh pipeline** (preprocessing + model), bukan hanya model |
| **joblib vs pickle** | `joblib` lebih efisien untuk array NumPy besar; gunakan `compress=3` untuk keseimbangan |
| **Model Metadata** | Simpan versi, waktu training, performa, dan hyperparameter untuk reproducibility |
| **Inference Consistency** | Pipeline memastikan preprocessing identik antara training dan production |
| **Model Versioning** | Gunakan semantic versioning (v1.0.0); simpan multiple model versions |
| **Monitoring** | Pantau distribusi data input dan performa model di production secara berkala |

### Production ML Workflow

```
1. Data Collection & Validation
      ↓
2. Feature Engineering & Preprocessing
      ↓
3. Model Training & Hyperparameter Tuning
      ↓
4. Model Evaluation (holdout test set)
      ↓
5. Pipeline Serialization (joblib.dump)
      ↓
6. Deployment (API / batch job)
      ↓
7. Monitoring & Retraining
```

### Best Practices untuk Production
- Unit test prediksi model setelah loading
- A/B testing sebelum full rollout
- Feature drift monitoring
- Retraining schedule berdasarkan model decay
- Fallback mechanism jika model gagal

**Nama: Fransisco Sitepu | NIM: 101032300004 | Kelas: TK 47 03 | Mata Kuliah: Machine Learning**
